# Pulse
SCMG Anomaly Detection — LSTM-based GNN pipeline

In [ ]:
# Ensure notebook runs from project root regardless of launch directory
import os, sys
_here = os.getcwd()
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Standard library
import pickle
import time
import subprocess
import smtplib
import yaml
from datetime import datetime, timedelta
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Third-party
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# PyTorch
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.data import Data, Batch

# Visualization
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import plotly.graph_objects as go

load_dotenv()

## `config/config.py`

In [ ]:
class Config:
    # Grab SCMG API and Sensitive Credentials (from .env)
    API_TOKEN = os.getenv("SCMG_API_TOKEN")
    API_BASE_URL = os.getenv("SCMG_API_BASE_URL", "https://www.strawberrycreek.org/api/creek-data/")

    # NWS Weather (Lawrence Berkeley National Lab Weather Data), this has no API key required, just a User-Agent)
    # Used to supplement the creek's own rain gauge with an independent measurement.
    # Set USE_NWS_RAIN=false in .env to disable.
    NWS_STATION_ID = os.getenv("NWS_STATION_ID", "LBNL1")
    NWS_USER_AGENT = os.getenv("NWS_USER_AGENT", "SCMG-AnDeSys/1.0")
    USE_NWS_RAIN   = os.getenv("USE_NWS_RAIN", "true").lower() == "true"

    # SQL Database (this is optional, it is only needed when using --data-source sql)
    SQL_CONNECTION_STRING = os.getenv("SQL_CONNECTION_STRING")
    SQL_TABLE_NAME        = os.getenv("SQL_TABLE_NAME", "creek_data")
    # Column names in your SQL table, you should override these if your schema differs
    SQL_TIMESTAMP_COL     = os.getenv("SQL_TIMESTAMP_COL", "timestamp")
    SQL_LOCATION_COL      = os.getenv("SQL_LOCATION_COL", "station_id")
    SQL_CONDUCTIVITY_COL  = os.getenv("SQL_CONDUCTIVITY_COL", "Meter_Hydros21_Cond")
    SQL_DEPTH_COL         = os.getenv("SQL_DEPTH_COL", "Meter_Hydros21_Depth")
    SQL_TEMP_COL          = os.getenv("SQL_TEMP_COL", "Meter_Hydros21_Temp")
    SQL_RAIN_COL          = os.getenv("SQL_RAIN_COL", "TE_TR_525USW_Precip_5minTotal")
    SQL_TEMP2M_COL        = os.getenv("SQL_TEMP2M_COL", "Sensirion_SHT40_Temperature")

    # Path Setup for YAML
    _current_dir = os.path.join(os.getcwd(), 'config')
    _yaml_path = os.path.join(_current_dir, 'settings.yaml')

    # Load YAML
    try:
        with open(_yaml_path, 'r') as f:
            _s = yaml.safe_load(f)
    except FileNotFoundError:
        print(f"Warning: settings.yaml not found at {_yaml_path}. Using hardcoded defaults.")
        _s = {}

    # Map YAML to Class Variables
    HIDDEN_DIM      = _s.get('model_architecture', {}).get('hidden_dim', 16)
    GNN_LAYERS      = _s.get('model_architecture', {}).get('gnn_layers', 1)
    TEMPORAL_LAYERS = _s.get('model_architecture', {}).get('temporal_layers', 1)
    DROPOUT         = _s.get('model_architecture', {}).get('dropout', 0.1)
    GNN_TYPE        = _s.get('model_architecture', {}).get('gnn_type', 'GCN')

    EPOCHS          = _s.get('training', {}).get('epochs', 10)
    BATCH_SIZE      = _s.get('training', {}).get('batch_size', 128)
    LEARNING_RATE   = _s.get('training', {}).get('learning_rate', 0.001)
    PATIENCE        = _s.get('training', {}).get('patience', 5)
    TRAIN_SPLIT     = _s.get('training', {}).get('train_split', 0.8)

    THRESHOLD_PERCENTILE      = _s.get('detection', {}).get('threshold_percentile', 99)
    RAIN_WINDOW_HOURS         = _s.get('detection', {}).get('rain_window_hours', 12)
    RAIN_THRESHOLD_MULTIPLIER = _s.get('detection', {}).get('rain_multiplier', 2.0)
    RAIN_AMOUNT_THRESHOLD     = _s.get('detection', {}).get('rain_amount_threshold', 0.1)

    IMPUTATION_LIMIT_HOURS = _s.get('preprocessing', {}).get('imputation_limit_hours', 3)

    # Constants
    SEQUENCE_LENGTH = 24
    DATA_FILE = 'full_creek_gnn.csv'
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    LOCATIONS = ['footbridge', 'north_fork_0', 'south_fork_2', 'south_fork_1', 'oxford']
    LOCATION_TO_IDX = {loc: idx for idx, loc in enumerate(LOCATIONS)}
    NUM_FEATURES = 10 # Adjust this based on the actual column count

## `src/ingest/api_client.py`

In [ ]:
def fetch_creek_data(site, start_time, end_time, variables=None):
    """
    Query the Strawberry Creek API for a specific site and time range.
    Returns a pandas DataFrame.
    """

    headers = {
        "Authorization": f"Token {Config.API_TOKEN}"
    }

    # Base parameters required by the GET endpoint
    params = {
        "site": site,
        "start": start_time,
        "end": end_time
    }

    # Handle optional variable filtering
    if variables:
        params["vars"] = variables

    try:
        response = requests.get(
            Config.API_BASE_URL,
            headers=headers,
            params=params,
            timeout=60
        )

        # Check for 401 Unauthorized or 404 Not Found
        if response.status_code != 200:
            print(f"Error: API returned status {response.status_code}")
            print(f"Details: {response.text}")
            return pd.DataFrame()

        data = response.json()

        if not data:
            print(f"No records found for {site} in the specified range.")
            return pd.DataFrame()

        df = pd.DataFrame(data)

        # Standardize timestamp for downstream processing
        if 'timestamp' in df.columns:
            df['timestamp'] = pd.to_datetime(df['timestamp'])

        return df

    except requests.exceptions.Timeout:
        print("The request timed out. The server might be busy.")
        return pd.DataFrame()
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return pd.DataFrame()

def fetch_network_snapshot(start_time, end_time):
    """
    Iterates through all sites defined in Config to pull a full dataset.
    Useful for building the graph-based training set.
    """
    frames = []

    for site in Config.LOCATIONS:
        print(f"Requesting data: {site}...")
        df_site = fetch_creek_data(site, start_time, end_time)

        if not df_site.empty:
            df_site['station_id'] = site
            frames.append(df_site)

    if not frames:
        print("No data retrieved for any site.")
        return pd.DataFrame()

    # Combine all sites into a single flat file for the DataLoader
    return pd.concat(frames, ignore_index=True)

## `src/ingest/sql_client.py`

In [ ]:
def _get_engine():
    if not Config.SQL_CONNECTION_STRING:
        raise ValueError(
            "SQL_CONNECTION_STRING is not set. "
            "Add it to your .env file before using --data-source sql."
        )
    return create_engine(Config.SQL_CONNECTION_STRING)


def fetch_creek_data_sql(site, start_time, end_time):
    """
    Query a SQL database for a specific site and time range.
    Returns a DataFrame with internal column names (conductivity, depth, etc.)
    plus 'timestamp' and 'station_id', mirroring the API client's output shape.
    """
    # Map the user-configured SQL column names to the internal names expected
    # by data_loader's column_mapping ('timestamp' -> 'datetime', 'station_id' -> 'location').
    sql_to_internal = {
        Config.SQL_CONDUCTIVITY_COL: "conductivity",
        Config.SQL_DEPTH_COL: "depth",
        Config.SQL_TEMP_COL: "temperature",
        Config.SQL_RAIN_COL: "rain_mm",
        Config.SQL_TEMP2M_COL: "Temp2m_Avg",
        Config.SQL_TIMESTAMP_COL: "timestamp",
        Config.SQL_LOCATION_COL: "station_id",
    }

    col_list = ", ".join(f'"{c}"' for c in sql_to_internal)

    query = text(
        f"SELECT {col_list} FROM {Config.SQL_TABLE_NAME} "  # noqa: S608
        f"WHERE {Config.SQL_LOCATION_COL} = :site "
        f"  AND {Config.SQL_TIMESTAMP_COL} >= :start_time "
        f"  AND {Config.SQL_TIMESTAMP_COL} <= :end_time "
        f"ORDER BY {Config.SQL_TIMESTAMP_COL}"
    )

    try:
        engine = _get_engine()
        with engine.connect() as conn:
            df = pd.read_sql(query, conn, params={
                "site": site,
                "start_time": start_time,
                "end_time": end_time,
            })

        if df.empty:
            print(f"No records found in SQL for {site} in the specified range.")
            return df

        df = df.rename(columns=sql_to_internal)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        return df

    except Exception as e:
        print(f"SQL query failed for site '{site}': {e}")
        return pd.DataFrame()


def fetch_network_snapshot_sql(start_time, end_time):
    """
    Iterates through all sites defined in Config and pulls a full dataset from SQL.
    Returns a combined DataFrame in the same shape as fetch_network_snapshot().
    """
    frames = []

    for site in Config.LOCATIONS:
        print(f"Querying SQL database: {site}...")
        df_site = fetch_creek_data_sql(site, start_time, end_time)
        if not df_site.empty:
            frames.append(df_site)

    if not frames:
        print("No data retrieved from SQL database for any site.")
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)

## `src/ingest/weather_client.py`

In [ ]:
def fetch_nws_precipitation(start_time, end_time):
    """
    Fetch hourly precipitation observations from the NWS for the configured station.

    Uses the free NWS observations API (no API key required, just a User-Agent).
    Returns a DataFrame with a UTC DatetimeIndex and a single 'rain_mm' column.
    Returns an empty DataFrame on any failure.
    """
    base_url = f"https://api.weather.gov/stations/{Config.NWS_STATION_ID}/observations"
    headers = {
        "User-Agent": Config.NWS_USER_AGENT,
        "Accept": "application/geo+json",
    }
    params = {
        "start": _ensure_utc_suffix(start_time),
        "end": _ensure_utc_suffix(end_time),
    }

    all_features = []
    next_url = base_url

    while next_url:
        try:
            resp = requests.get(
                next_url,
                headers=headers,
                params=(params if next_url == base_url else None),
                timeout=30,
            )
            if resp.status_code != 200:
                print(f"NWS API error {resp.status_code} for station {Config.NWS_STATION_ID}: {resp.text[:200]}")
                break
            data = resp.json()
            all_features.extend(data.get("features", []))
            next_url = data.get("pagination", {}).get("next")
        except Exception as e:
            print(f"NWS fetch failed: {e}")
            break

    if not all_features:
        print(f"No NWS observations returned for station {Config.NWS_STATION_ID}.")
        return pd.DataFrame()

    records = []
    for feat in all_features:
        props = feat.get("properties", {})
        ts = props.get("timestamp")
        precip = props.get("precipitationLastHour") or {}
        val = precip.get("value")  # NWS reports in meters; None = not observed
        if ts is not None:
            records.append({
                "datetime": ts,
                # Convert m to mm, keep None to distinguish no report from actual zero rain
                "rain_mm": val * 1000.0 if val is not None else None,
            })

    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
    df = df.set_index("datetime").sort_index()
    df["rain_mm"] = pd.to_numeric(df["rain_mm"], errors="coerce")
    return df


def _ensure_utc_suffix(t):
    """Make sure a timestamp string is recognisable as UTC by the NWS API."""
    s = str(t)
    if not (s.endswith("Z") or "+00" in s or "-00" in s):
        return s + "Z"
    return s

## `src/ingest/data_loader.py`

In [ ]:
def load_and_preprocess_data(file_path=Config.DATA_FILE, force_download=False, days=30, data_source="api"):
    """
    Loads creek data. If the local file is missing or force_download is True,
    it fetches data from the configured source for the specified number of days.

    data_source: "api" (default) pulls from the Strawberry Creek REST API.
                 "sql" pulls from the SQL database configured in .env.
    """

    if not os.path.exists(file_path) or force_download:
        source_label = "API" if data_source == "api" else "SQL database"
        if force_download:
            print(f"Refresh requested. Fetching last {days} days of data from {source_label}...")
        else:
            print(f"Local file '{file_path}' not found. Initializing {source_label} pull...")

        end_date = datetime.now()
        start_date = end_date - timedelta(days=days)

        if data_source == "sql":
            df_raw = fetch_network_snapshot_sql(
                start_time=start_date.isoformat(),
                end_time=end_date.isoformat(),
            )
        else:
            df_raw = fetch_network_snapshot(
                start_time=start_date.isoformat(),
                end_time=end_date.isoformat(),
            )

        if df_raw.empty:
            print(f"CRITICAL ERROR: No data retrieved from {source_label}.")
            if not os.path.exists(file_path):
                sys.exit(1)
            print("Falling back to existing local file.")
        else:
            # Map raw source column names to the internal names used by the GNN.
            # For the SQL path the sensor columns are already renamed by sql_client;
            # only 'timestamp' -> 'datetime' and 'station_id' -> 'location' remain.
            column_mapping = {
                'Meter_Hydros21_Cond': 'conductivity',
                'Meter_Hydros21_Depth': 'depth',
                'Meter_Hydros21_Temp': 'temperature',
                'TE_TR_525USW_Precip_5minTotal': 'rain_mm',
                'Sensirion_SHT40_Temperature': 'Temp2m_Avg',
                'timestamp': 'datetime',
                'station_id': 'location',
            }

            df_raw = df_raw.rename(columns=column_mapping)

            if Config.USE_NWS_RAIN:
                df_raw = _merge_nws_rain(df_raw, start_date, end_date)

            df_raw.to_csv(file_path, index=False)
            print(f"Data successfully cached to {file_path}")

    # Load from the cached or existing file
    df = pd.read_csv(file_path)

    # Preprocessing and Indexing
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
    df = df.set_index('datetime').sort_index()

    print(f"--- Data Loading Report ---")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Range: {df.index.min()} to {df.index.max()}")

    # Pass all numeric columns through instead of a hardcoded list.
    # New parameters from the API or SQL get picked up automatically.
    # Only rain_mm and depth have special meaning downstream (rain flags
    # and absent sensor detection). Everything else is just a feature.
    _non_feature = {'Unnamed: 0', 'delta', 'Precip_Max', 'EnviroDIY_Mayfly_Batt'}
    feature_cols = [
        col for col in df.columns
        if col != 'location'
        and col not in _non_feature
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    if not feature_cols:
        print("CRITICAL ERROR: No numeric feature columns found in dataset.")
        sys.exit(1)

    df_featured = df[['location'] + feature_cols].copy()

    print(f"Active features ({len(feature_cols)}): {', '.join(feature_cols)}")
    if 'rain_mm' not in feature_cols:
        print("Warning: 'rain_mm' not found — rain-adjusted detection will be disabled.")
    print(f"---------------------------\n")

    return df_featured, df, Config.LOCATIONS


def _merge_nws_rain(df_raw, start_date, end_date):
    """
    Fetch NWS hourly precipitation for the same window and merge it into df_raw.

    Strategy: take max(creek_rain_mm, nws_rain_mm) at each timestamp so that
    if the creek's tipping-bucket gauge fails or reads zero during a real storm,
    the NWS reading fills the gap.  NWS values (hourly) are aligned by flooring
    each creek timestamp to the nearest hour.
    """
    print(f"Fetching NWS precipitation for station {Config.NWS_STATION_ID}...")
    df_nws = fetch_nws_precipitation(start_date.isoformat(), end_date.isoformat())

    if df_nws.empty:
        print("NWS rain data unavailable — using creek sensor rain only.")
        return df_raw

    # Align: floor each creek timestamp to the hour, then look up the NWS
    # hourly observation for that hour (precipitationLastHour covers the
    # preceding 60 minutes, so the 14:00 reading = rain from 13:00–14:00).
    creek_dt = pd.to_datetime(df_raw["datetime"], utc=True)
    hourly_keys = creek_dt.dt.floor("h")

    nws_lookup = df_nws["rain_mm"].fillna(0)
    nws_per_row = hourly_keys.map(nws_lookup).fillna(0).values

    creek_rain = pd.to_numeric(df_raw.get("rain_mm", 0), errors="coerce").fillna(0).values
    df_raw["rain_mm"] = pd.Series(
        [max(c, n) for c, n in zip(creek_rain, nws_per_row)],
        index=df_raw.index,
    )

    nws_contributed = int((nws_per_row > creek_rain).sum())
    print(f"NWS rain merged — supplemented {nws_contributed} rows where creek gauge read lower.")
    return df_raw

## `src/preprocessing/data_processor.py`

In [ ]:
def _impute_short_gaps(df, feature_cols, limit_hours):
    """
    For each (location, feature) pair, linearly interpolate over gaps shorter
    than limit_hours.  Longer gaps are left as NaN so the validity check
    downstream still excludes them.  limit_area='inside' prevents extrapolation
    beyond the first/last real observation.
    """
    timestamps = df.index.sort_values().unique()
    if len(timestamps) < 2:
        return df

    median_interval = pd.Series(timestamps.astype('int64')).diff().dropna().median()
    median_interval = pd.Timedelta(median_interval)
    rows_per_hour = pd.Timedelta('1h') / median_interval
    limit_rows = max(1, int(limit_hours * rows_per_hour))

    print(f"--- Imputing Short Sensor Gaps (limit: {limit_hours}h / {limit_rows} rows) ---")
    df = df.copy()
    total_filled = 0

    for location in df['location'].unique():
        mask = df['location'] == location
        before = int(df.loc[mask, feature_cols].isna().sum().sum())

        df.loc[mask, feature_cols] = (
            df.loc[mask, feature_cols]
            .interpolate(method='time', limit=limit_rows, limit_area='inside')
        )

        filled = before - int(df.loc[mask, feature_cols].isna().sum().sum())
        if filled > 0:
            print(f"  [{location}] filled {filled} missing values")
        total_filled += filled

    print(f"[INFO] Imputation complete — {total_filled} values filled across all locations.\n")
    return df


def prepare_sequences_normalized(df_featured, location_to_idx, sequence_length=Config.SEQUENCE_LENGTH):
    """
    Prepare temporal sequences with Z-score normalization.
    Short sensor gaps are interpolated before normalization.
    Oxford's permanently absent depth channel is zeroed so the model learns
    to ignore it rather than treat it as signal.
    """

    # Identify feature columns
    exclude_cols = ['location', 'Unnamed: 0', 'delta', 'Precip_Max', 'EnviroDIY_Mayfly_Batt']
    feature_cols = [col for col in df_featured.select_dtypes(include=[np.number]).columns.tolist()
                    if col not in exclude_cols]

    print(f"[INFO] Using {len(feature_cols)} features: {', '.join(feature_cols)}")

    # Fill short sensor-dropout gaps before anything else.
    # Gaps longer than IMPUTATION_LIMIT_HOURS remain NaN and are caught by the
    # validity check below, so they never enter a training sequence.
    df_featured = _impute_short_gaps(df_featured, feature_cols, Config.IMPUTATION_LIMIT_HOURS)

    # Find (node, feature) pairs where a sensor just doesn't exist at a location
    # (e.g. Oxford has no depth sensor). Zero those out in every sequence so the
    # model sees a consistent value, and skip them in the NaN validity check so
    # those timesteps aren't thrown out. Works for any absent sensor, not just Oxford.
    print("--- Detecting Permanently Absent Sensor Channels ---")
    absent_pairs = set()  # {(node_idx, feat_idx), ...}
    for location, node_idx in location_to_idx.items():
        loc_mask = df_featured['location'] == location
        for feat_idx, feat in enumerate(feature_cols):
            if df_featured.loc[loc_mask, feat].isna().all():
                absent_pairs.add((node_idx, feat_idx))
                print(f"  {location}/{feat}: no data — will be zeroed in sequences")
    if not absent_pairs:
        print("  (none)")
    print()

    # Z-score normalization.
    # Fit only on rows that are fully valid so NaN values from long outages
    # don't corrupt the scaler's mean/std statistics.
    all_data = []
    for location in location_to_idx.keys():
        loc_data = df_featured[df_featured['location'] == location][feature_cols].values
        all_data.append(loc_data)

    all_data = np.vstack(all_data)
    valid_rows = ~np.isnan(all_data).any(axis=1)
    scaler = StandardScaler()
    scaler.fit(all_data[valid_rows])

    # Apply normalization to a copy of the dataframe
    df_normalized = df_featured.copy()
    for location in location_to_idx.keys():
        loc_mask = df_featured['location'] == location
        df_normalized.loc[loc_mask, feature_cols] = scaler.transform(
            df_featured.loc[loc_mask, feature_cols].values
        )

    # Pivot to 3D array: (timesteps, nodes, features)
    print("--- Building 3D Array ---")
    timestamps_all = sorted(df_normalized.index.unique())
    num_nodes = len(location_to_idx)
    num_features = len(feature_cols)

    data_3d = np.full((len(timestamps_all), num_nodes, num_features), np.nan)

    for t_idx, timestamp in enumerate(tqdm(timestamps_all, desc="Pivoting data")):
        t_data = df_normalized.loc[timestamp]

        # Only process if we have data for all nodes at this timestamp
        if len(t_data) == num_nodes:
            for _, row in t_data.iterrows():
                node_idx = location_to_idx[row['location']]
                data_3d[t_idx, node_idx, :] = row[feature_cols].values

    # Validate timesteps, but skip permanently absent channels in the NaN check
    def is_valid_timestep(t_idx):
        t_data = data_3d[t_idx].copy()
        for node_idx, feat_idx in absent_pairs:
            t_data[node_idx, feat_idx] = 0
        return not np.isnan(t_data).any()

    valid_mask = np.array([is_valid_timestep(i) for i in range(len(timestamps_all))])
    print(f"[INFO] Valid timesteps: {valid_mask.sum():,} / {len(valid_mask):,}")

    # Create sliding window sequences
    print(f"--- Creating Sequences (Length: {sequence_length}) ---")
    sequences = []
    targets = []
    sequence_timestamps = []

    for i in tqdm(range(len(timestamps_all) - sequence_length), desc="Sliding window"):
        # Ensure the entire window + the target step are valid
        if valid_mask[i:i+sequence_length+1].all():
            seq = data_3d[i:i+sequence_length].copy()
            target = data_3d[i+sequence_length].copy()

            # Zero out permanently absent channels so the model doesn't see NaN
            for node_idx, feat_idx in absent_pairs:
                seq[:, node_idx, feat_idx] = 0
                target[node_idx, feat_idx] = 0

            sequences.append(seq)
            targets.append(target)
            sequence_timestamps.append(timestamps_all[i+sequence_length])

    print(f"[INFO] Final Sequence Count: {len(sequences):,}")

    return np.array(sequences), np.array(targets), sequence_timestamps, scaler, feature_cols

## `src/utils/graph_utils.py`

In [ ]:
def _create_edge_index(edges, location_to_idx):
    """
    Helper function to convert an edge list to PyTorch Geometric format.
    Kept separate for modularity and testing.
    """
    edge_list = [[location_to_idx[src], location_to_idx[dst]] for src, dst in edges]
    # .t() transposes to [2, num_edges] format required by PyTorch Geometric
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    return edge_index

def create_graph_topology():
    """
    Define the creek network as a directed graph and create the edge index.
    Matches the Colab flow topology exactly.
    """
    locations = Config.LOCATIONS
    location_to_idx = Config.LOCATION_TO_IDX

    # Defining creek flow topology (Identical to your Colab logic)
    # This will represent the physical direction of water flow
    edges = [
        ('north_fork_0', 'footbridge'),
        ('footbridge', 'oxford'),
        ('south_fork_1', 'south_fork_2'),
        ('south_fork_2', 'oxford'),
    ]

    # Helper to create the tensor and move to the configured device
    edge_index = _create_edge_index(edges, location_to_idx).to(Config.DEVICE)

    print(f"--- Graph Topology Report ---")
    print(f"Nodes: {len(locations)} sensors")
    print(f"Edges: {len(edges)} flow connections")
    print(f"Device: {edge_index.device}")
    print(f"-----------------------------\n")

    return edge_index, locations, location_to_idx

## `src/models/model.py`

In [ ]:
class TemporalGNN(nn.Module):
    """
    Temporal Graph Neural Network for creek sensor time-series prediction.
    Combines spatial (GNN) and temporal (LSTM) feature extraction.
    """

    def __init__(self, num_node_features):
        super(TemporalGNN, self).__init__()

        # Pull architecture settings directly from Config
        self.hidden_dim = Config.HIDDEN_DIM
        self.gnn_type = Config.GNN_TYPE
        gnn_layers = Config.GNN_LAYERS
        temporal_layers = Config.TEMPORAL_LAYERS
        dropout = Config.DROPOUT

        # Spatial layers (GNN)
        self.gnn_layers = nn.ModuleList()

        if self.gnn_type == 'GCN':
            self.gnn_layers.append(GCNConv(num_node_features, self.hidden_dim))
            for _ in range(gnn_layers - 1):
                self.gnn_layers.append(GCNConv(self.hidden_dim, self.hidden_dim))
        elif self.gnn_type == 'GAT':
            # GAT often uses multiple attention heads
            self.gnn_layers.append(GATConv(num_node_features, self.hidden_dim, heads=4, concat=True))
            for _ in range(gnn_layers - 1):
                self.gnn_layers.append(GATConv(self.hidden_dim * 4, self.hidden_dim))

        # Temporal layer (LSTM)
        lstm_input_dim = self.hidden_dim * 4 if self.gnn_type == 'GAT' else self.hidden_dim
        self.lstm = nn.LSTM(
            input_size=lstm_input_dim,
            hidden_size=self.hidden_dim,
            num_layers=temporal_layers,
            batch_first=True,
            dropout=dropout if temporal_layers > 1 else 0
        )

        # Output layer (Predicts next timestep features for each node)
        self.output_layer = nn.Linear(self.hidden_dim, num_node_features)

        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()

    def forward(self, x_sequence, edge_index, batch_size, num_nodes):
        """
        Forward pass optimized for PyTorch Geometric Batching.
        """
        batch_size, seq_len, num_nodes, num_features = x_sequence.shape

        # Pre-process the edge index for the entire batch once
        # This is much faster than recreating it inside the loop
        data_list = [Data(x=torch.zeros(num_nodes, num_features), edge_index=edge_index) for _ in range(batch_size)]
        batch_loader = Batch.from_data_list(data_list).to(x_sequence.device)
        batch_edge_index = batch_loader.edge_index

        gnn_outputs = []

        # Process each timestep through spatial layers
        for t in range(seq_len):
            x_t = x_sequence[:, t, :, :]  # (batch_size, num_nodes, num_features)
            x_flat = x_t.reshape(-1, num_features)  # Flatten for batch GNN processing

            h = x_flat
            for gnn_layer in self.gnn_layers:
                h = gnn_layer(h, batch_edge_index)
                h = self.activation(h)
                h = self.dropout(h)

            # Reshape back to batch and pool across nodes to get graph-level context
            h = h.reshape(batch_size, num_nodes, -1)
            h_graph = h.mean(dim=1)  # (batch_size, hidden_dim)
            gnn_outputs.append(h_graph)

        # Temporal Processing
        temporal_input = torch.stack(gnn_outputs, dim=1) # (batch_size, seq_len, hidden_dim)
        lstm_out, _ = self.lstm(temporal_input)

        # Take the last hidden state of the LSTM
        last_hidden = lstm_out[:, -1, :]

        # Node-level Prediction
        # Expand the temporal context back to each node
        expanded = last_hidden.unsqueeze(1).expand(-1, num_nodes, -1)
        predictions = self.output_layer(expanded)

        return predictions

## `src/training/trainer.py`

In [ ]:
def train_temporal_gnn(
    model,
    train_sequences,
    train_targets,
    edge_index,
    val_sequences=None,
    val_targets=None,
    epochs=Config.EPOCHS,
    batch_size=Config.BATCH_SIZE,
    learning_rate=Config.LEARNING_RATE,
    patience=Config.PATIENCE,
    device=Config.DEVICE
):
    """
    Train temporal GNN with mixed-precision acceleration.
    Learns baseline creek physics via reconstruction (MSE loss).
    """
    model = model.to(device)
    edge_index = edge_index.to(device)
    device_type = 'cuda' if 'cuda' in str(device) else 'cpu'

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    scaler = GradScaler(device_type=device_type)

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    train_losses = []
    val_losses = []

    print(f"--- Starting Training on {device_type.upper()} ---")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        num_batches = 0

        # Mini-batch training
        for i in range(0, len(train_sequences), batch_size):
            # Convert to tensors inside the loop to save GPU memory
            batch_seq = torch.FloatTensor(train_sequences[i:i+batch_size]).to(device)
            batch_target = torch.FloatTensor(train_targets[i:i+batch_size]).to(device)

            optimizer.zero_grad()

            # Mixed precision forward pass
            with autocast(device_type=device_type):
                predictions = model(
                    batch_seq,
                    edge_index,
                    batch_size=len(batch_seq),
                    num_nodes=batch_seq.shape[2]
                )
                loss = criterion(predictions, batch_target)

            # Mixed precision backward pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            num_batches += 1

        avg_train_loss = epoch_loss / num_batches
        train_losses.append(avg_train_loss)

        # Validation logic
        if val_sequences is not None:
            model.eval()
            with torch.no_grad(), autocast(device_type=device_type):
                val_seq = torch.FloatTensor(val_sequences).to(device)
                val_tgt = torch.FloatTensor(val_targets).to(device)

                val_pred = model(
                    val_seq,
                    edge_index,
                    batch_size=len(val_seq),
                    num_nodes=val_seq.shape[2]
                )

                val_loss = criterion(val_pred, val_tgt).item()
                val_losses.append(val_loss)

                # Track best model for early stopping
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_model_state = model.state_dict().copy()
                    patience_counter = 0
                else:
                    patience_counter += 1

                if patience_counter >= patience:
                    print(f"[STATUS] Early stopping triggered at epoch {epoch+1}")
                    break

        # Progress logging every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == 0:
            status = f"Epoch {epoch+1:3d}/{epochs} | Train Loss: {avg_train_loss:.6f}"
            if val_sequences is not None:
                status += f" | Val Loss: {val_loss:.6f}"
            print(status)

    # Restore best weights if available
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"[INFO] Restored model weights from epoch with Val Loss: {best_val_loss:.6f}")

    print("--- Training Complete ---\n")
    return train_losses, val_losses

## `src/anomalies/anomaly_detector.py`

In [ ]:
def compute_anomaly_scores(model, sequences, targets, edge_index, device):
    """
    Compute reconstruction errors (anomaly scores) for sequences.

    Returns:
        errors: np.ndarray (absolute prediction error per sensor per feature)
        predictions: np.ndarray (raw model outputs)
    """
    model.eval()
    model = model.to(device)
    edge_index = edge_index.to(device)

    with torch.no_grad():
        # Convert to tensor and move to device
        seq_tensor = torch.FloatTensor(sequences).to(device)
        target_tensor = torch.FloatTensor(targets).to(device)

        predictions = model(
            seq_tensor,
            edge_index,
            batch_size=len(sequences),
            num_nodes=sequences.shape[2]
        )

        # Compute absolute reconstruction error (MAE)
        errors = torch.abs(predictions - target_tensor)

    return errors.cpu().numpy(), predictions.cpu().numpy()

def detect_spills_with_rain_adjustment(
    system_anomaly_scores,
    timestamps,
    df_original,
    locations,
    threshold_percentile=Config.THRESHOLD_PERCENTILE,
    rain_window_hours=Config.RAIN_WINDOW_HOURS,
    rain_threshold_multiplier=Config.RAIN_THRESHOLD_MULTIPLIER,
    rain_amount_threshold=Config.RAIN_AMOUNT_THRESHOLD
):
    """
    Rain-aware spill detection logic with adaptive thresholding.

    How it works:
    First, it'll identify 'Rain-Affected' periods using a lookback window.
    Next, it'll calculate a base threshold from the anomaly score distribution.
    Last, it'll scale the threshold up during rain to account for natural runoff noise.
    """
    # Skip rain adjustment if there's no rain_mm column or it's all NaN.
    if 'rain_mm' not in df_original.columns or df_original['rain_mm'].isna().all():
        print("[INFO] No rain_mm data available — running without rain adjustment.")
        base_threshold = np.percentile(system_anomaly_scores, threshold_percentile)
        adjusted_thresholds = np.full(len(timestamps), base_threshold)
        spill_flags = system_anomaly_scores > adjusted_thresholds
        rain_flags = np.zeros(len(timestamps), dtype=bool)
        print(f"--- Detection Summary ---")
        print(f"Total Spills Detected: {spill_flags.sum()}")
        print(f"Rain-Affected Spills: 0 (no rain data)")
        print(f"Dry-Weather Spills: {spill_flags.sum()}")
        print(f"-------------------------\n")
        return spill_flags, rain_flags, adjusted_thresholds

    # Use the first location as the weather reference for rain data
    rain_data = df_original[df_original['location'] == locations[0]][['rain_mm']].copy()

    # Now identify rain-affected periods (Vectorized approach where possible)
    rain_flags = np.zeros(len(timestamps), dtype=bool)

    # We use a rolling window logic to see if it rained in the last X hours
    for i, ts in enumerate(timestamps):
        lookback_start = ts - pd.Timedelta(hours=rain_window_hours)
        recent_rain = rain_data[(rain_data.index >= lookback_start) & (rain_data.index <= ts)]

        if recent_rain['rain_mm'].sum() > rain_amount_threshold:
            rain_flags[i] = True

    # Calculate Adaptive Thresholds
    base_threshold = np.percentile(system_anomaly_scores, threshold_percentile)

    # Apply the multiplier only where rain_flags is True
    adjusted_thresholds = np.where(
        rain_flags,
        base_threshold * rain_threshold_multiplier,
        base_threshold
    )

    # Generate Spill Flags
    spill_flags = system_anomaly_scores > adjusted_thresholds

    # Summary Logging
    print(f"--- Detection Summary ---")
    print(f"Total Spills Detected: {spill_flags.sum()}")
    print(f"Rain-Affected Spills: {(spill_flags & rain_flags).sum()}")
    print(f"Dry-Weather Spills: {(spill_flags & ~rain_flags).sum()}")
    print(f"-------------------------\n")

    return spill_flags, rain_flags, adjusted_thresholds

## `src/utils/visualizations.py`

In [ ]:
def plot_static_dashboard(
    timestamps,
    system_anomaly_scores,
    normalized_anomaly_scores,
    adjusted_thresholds,
    base_threshold,
    spill_flags,
    rain_flags,
    df_original,
    locations,
    threshold_percentile
):
    """
    Generates and SAVES the high-resolution Matplotlib dashboard.
    """
    spills_during_rain = spill_flags & rain_flags
    spills_no_rain = spill_flags & ~rain_flags

    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)

    # Main anomaly plot
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(timestamps, system_anomaly_scores, linewidth=1, color='blue', alpha=0.7, label='Anomaly score')
    ax1.plot(timestamps, adjusted_thresholds, color='darkorange', linestyle='--', linewidth=2.5, label='Adjusted threshold')
    ax1.axhline(base_threshold, color='gold', linestyle=':', linewidth=2, label=f'Base ({threshold_percentile}th %ile)')

    # Shade rain
    for i, (ts, is_rain) in enumerate(zip(timestamps, rain_flags)):
        if is_rain:
            ax1.axvspan(ts, ts + pd.Timedelta(minutes=15), alpha=0.1, color='cyan')

    # Scatter spills
    ax1.scatter([timestamps[i] for i in range(len(timestamps)) if spills_during_rain[i]],
                [system_anomaly_scores[i] for i in range(len(timestamps)) if spills_during_rain[i]],
                color='purple', s=50, zorder=5, label='Spill (rain)', marker='^')

    ax1.scatter([timestamps[i] for i in range(len(timestamps)) if spills_no_rain[i]],
                [system_anomaly_scores[i] for i in range(len(timestamps)) if spills_no_rain[i]],
                color='red', s=50, zorder=5, label='Spill (no rain)', marker='o')

    ax1.set_title('Rain-Aware Spill Detection', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)

    # Rain timeline
    ax2 = fig.add_subplot(gs[1, :])
    # Filter by time range and location to get rain data
    mask = (df_original.index >= timestamps[0]) & (df_original.index <= timestamps[-1]) & (df_original['location'] == locations[0])
    rain_ts = df_original.loc[mask]['rain_mm']
    ax2.bar(rain_ts.index, rain_ts.values, width=0.01, color='cyan', alpha=0.7)
    ax2.set_ylabel('Rain (mm)', fontweight='bold')

    # Sensor contributions
    ax3 = fig.add_subplot(gs[2, 0])
    sensor_contributions = [normalized_anomaly_scores[spill_flags, loc_idx, 0].mean() if np.any(spill_flags) else 0
                            for loc_idx in range(len(locations))]
    ax3.barh(locations, sensor_contributions, color='orange')
    ax3.set_title('Sensor Contributions', fontsize=12, fontweight='bold')

    # Raw conductivity
    ax4 = fig.add_subplot(gs[2, 1])
    for location in locations:
        loc_data = df_original[df_original['location'] == location]
        ax4.plot(loc_data.index, loc_data['conductivity'], linewidth=1, alpha=0.6, label=location)
    ax4.set_title('Raw Conductivity Data', fontsize=12, fontweight='bold')

    plt.suptitle('SCMG System Results', fontsize=16, fontweight='bold', y=0.995)

    # --- SAVE LOGIC ---
    report_dir = "reports"
    if not os.path.exists(report_dir):
        os.makedirs(report_dir)

    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Save a history file and a 'latest' file for easy checking
    plt.savefig(os.path.join(report_dir, f"report_{timestamp_str}.png"), bbox_inches='tight', dpi=150)
    plt.savefig(os.path.join(report_dir, "latest_report.png"), bbox_inches='tight', dpi=150)

    print(f"[{datetime.now().strftime('%H:%M:%S')}] Dashboard saved to {report_dir}/")

    # Clean up memory
    plt.close('all')


def plot_interactive_plotly(
    timestamps,
    system_anomaly_scores,
    adjusted_thresholds,
    base_threshold,
    spill_flags,
    rain_flags,
    rain_threshold_multiplier,
    rain_window_hours,
    threshold_percentile
):
    """
    Saves the interactive Plotly visualization as an HTML file.
    Note: plotly.show() usually won't work in a background loop.
    """
    fig_plotly = go.Figure()

    fig_plotly.add_trace(go.Scatter(x=timestamps, y=system_anomaly_scores, name='Anomaly Score', line=dict(color='royalblue')))
    fig_plotly.add_trace(go.Scatter(x=timestamps, y=adjusted_thresholds, name='Adjusted Threshold', line=dict(color='darkorange', dash='dash')))

    spills_during_rain = spill_flags & rain_flags
    spills_no_rain = spill_flags & ~rain_flags

    fig_plotly.add_trace(go.Scatter(
        x=[timestamps[i] for i in range(len(timestamps)) if spills_during_rain[i]],
        y=[system_anomaly_scores[i] for i in range(len(timestamps)) if spills_during_rain[i]],
        mode='markers', name='Rain Spill', marker=dict(color='purple', symbol='triangle-up')
    ))

    fig_plotly.update_layout(
        title=f'Interactive Spill Detection (Multiplier={rain_threshold_multiplier}x)',
        template='plotly_white',
        height=600
    )

    # Save as HTML so we can open it in a browser later
    report_dir = "reports"
    if not os.path.exists(report_dir):
        os.makedirs(report_dir)

    fig_plotly.write_html(os.path.join(report_dir, "interactive_dashboard.html"))

## `src/utils/notifier.py`

In [ ]:
def send_spill_alert(spill_count, locations_affected):
    sender = os.getenv("ALERT_EMAIL_SENDER")
    password = os.getenv("ALERT_EMAIL_PASSWORD")
    receiver = os.getenv("ALERT_EMAIL_RECEIVER")

    if not all([sender, password, receiver]):
        print("Alert skipped: Email credentials missing in .env")
        return

    subject = f" ALERT: {spill_count} Potential Spill(s) Detected in Strawberry Creek"

    body = f"""
    The SCMG Anomaly Detection System has identified potential spills.

    Count: {spill_count}
    Affected Locations: {', '.join(locations_affected)}
    Timestamp: {os.popen('date').read()}

    Please check the latest dashboard visualization for details.
    """

    msg = MIMEMultipart()
    msg['From'] = sender
    msg['To'] = receiver
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))

    try:
        server = smtplib.SMTP(os.getenv("SMTP_SERVER"), int(os.getenv("SMTP_PORT")))
        server.starttls()
        server.login(sender, password)
        server.sendmail(sender, receiver, msg.as_string())
        server.quit()
        print("Spill alert email sent successfully.")
    except Exception as e:
        print(f"Failed to send email alert: {e}")

## `main.py`

In [ ]:
def main(mode="update", data_source="api", visualize=False):
    """
    Control logic for the GNN pipeline.
    """
    model_dir = "models"
    model_path = os.path.join(model_dir, "gnn_weights.pt")

    if not os.path.exists(model_dir):
        os.makedirs(model_dir)

    print("--- SCMG Anomaly Detection System ---")
    print(f"Execution Mode: {mode.upper()}")
    print(f"Device: {Config.DEVICE}")

    # DATA LOADING LOGIC
    if mode == "inference":
        # Pull only 2 days for speed during live monitoring
        df_featured, df_original, locations = load_and_preprocess_data(
            force_download=True, days=2, data_source=data_source
        )
    else:
        # Pull 30 days for training/updating
        df_featured, df_original, locations = load_and_preprocess_data(
            force_download=False, days=30, data_source=data_source
        )
    # ----------------------------------

    edge_index, _, location_to_idx = create_graph_topology()

    sequences, targets, timestamps, scaler, feature_cols = prepare_sequences_normalized(
        df_featured,
        location_to_idx,
        Config.SEQUENCE_LENGTH
    )

    if mode == "inference":
        train_seq, train_tgt = None, None
        test_seq, test_tgt = sequences, targets
        test_timestamps = timestamps
    else:
        split_idx = int(len(sequences) * Config.TRAIN_SPLIT)
        train_seq, test_seq = sequences[:split_idx], sequences[split_idx:]
        train_tgt, test_tgt = targets[:split_idx], targets[split_idx:]
        test_timestamps = timestamps[split_idx:]

    num_node_features = sequences.shape[3]
    model = TemporalGNN(num_node_features=num_node_features).to(Config.DEVICE)

    if mode in ["update", "inference"]:
        if os.path.exists(model_path):
            print(f"Loading weights from {model_path}")
            model.load_state_dict(torch.load(model_path, map_location=Config.DEVICE))
        else:
            print(f"No weight file found at {model_path}. Switching to fresh train.")
            mode = "train"

    if mode in ["train", "update"]:
        print("Commencing model optimization...")
        train_temporal_gnn(
            model,
            train_seq,
            train_tgt,
            edge_index,
            val_sequences=test_seq,
            val_targets=test_tgt
        )
        torch.save(model.state_dict(), model_path)
        print(f"Optimization complete. Weights saved to {model_path}")
        metadata_path = os.path.join(model_dir, "model_metadata.pkl")
        with open(metadata_path, "wb") as f:
            pickle.dump({
                "scaler": scaler,
                "feature_cols": feature_cols,
                "location_to_idx": location_to_idx,
            }, f)
        print(f"Model metadata saved to {metadata_path}")
    else:
        print("Skipping training phase. Entering evaluation mode.")

    model.eval()
    errors, predictions = compute_anomaly_scores(
        model,
        test_seq,
        test_tgt,
        edge_index,
        Config.DEVICE
    )

    system_scores = np.mean(errors, axis=(1, 2))

    spill_flags, rain_flags, adjusted_thresholds = detect_spills_with_rain_adjustment(
        system_anomaly_scores=system_scores,
        timestamps=test_timestamps,
        df_original=df_original,
        locations=locations
    )

    base_threshold = np.percentile(system_scores, Config.THRESHOLD_PERCENTILE)
    spill_count = np.sum(spill_flags)
    print(f"Detection cycle finished. Anomalies identified: {spill_count}")

    if visualize:
        plot_static_dashboard(
            timestamps=test_timestamps,
            system_anomaly_scores=system_scores,
            normalized_anomaly_scores=errors,
            adjusted_thresholds=adjusted_thresholds,
            base_threshold=base_threshold,
            spill_flags=spill_flags,
            rain_flags=rain_flags,
            df_original=df_original,
            locations=locations,
            threshold_percentile=Config.THRESHOLD_PERCENTILE
        )
        if mode != "inference":
            plot_interactive_plotly(
                timestamps=test_timestamps,
                system_anomaly_scores=system_scores,
                adjusted_thresholds=adjusted_thresholds,
                base_threshold=base_threshold,
                spill_flags=spill_flags,
                rain_flags=rain_flags,
                rain_threshold_multiplier=Config.RAIN_THRESHOLD_MULTIPLIER,
                rain_window_hours=Config.RAIN_WINDOW_HOURS,
                threshold_percentile=Config.THRESHOLD_PERCENTILE
            )

    if mode == "inference" and spill_count > 0:
        try:
            # Identify which locations had flags
            # Find any timestamp where a spill occurred, then find which sensors were above threshold
            affected_indices = np.unique(np.where(spill_flags)[0])
            # For simplicity in alerts, we notify based on nodes that peaked
            loc_peaks = np.any(spill_flags, axis=0)
            affected_locations = [locations[i] for i, peak in enumerate(loc_peaks) if peak]

            send_spill_alert(int(spill_count), affected_locations)
        except Exception as e:
            print(f"Alerting failed: {e}")

## Run

Set `mode`, `data_source`, and `visualize` as needed, then run this cell.

In [ ]:
# mode options:    'train' | 'update' | 'inference'
# data_source options: 'api' | 'sql'
main(mode='update', data_source='api', visualize=False)

## `run_live.py`

In [ ]:
def run_monitor():
    """
    Infinite loop that triggers the GNN inference every 15 minutes.
    """
    print("--- SCMG Live Monitoring Service Started ---")

    while True:
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\n[{now}] Starting detection cycle :)")

        try:
            # This calls main.py using the 'inference' mode
            # We use subprocess so each run starts with a clean memory state
            subprocess.run([sys.executable, "main.py", "--mode", "inference"], check=True)

        except subprocess.CalledProcessError as e:
            print(f"[{now}] Error during detection cycle: {e}")

        print(f"[{now}] Cycle complete. Sleeping for 15 minutes n^n")

        # 900 seconds = 15 minutes
        time.sleep(900)

In [ ]:
# Uncomment to start the live monitoring loop (runs indefinitely until interrupted)
# run_monitor()